# HC02 runner

Clones the repo, builds `blatt02/`, runs all sweeps on the Colab GPU (T4) and offers `results/` as a zip download.

Runtime > Change runtime type > **T4 GPU**, then Run all.

In [ ]:
# 1. Assert a GPU is present
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0, "No GPU runtime! Select Runtime > Change runtime type > T4 GPU."

In [ ]:
# 2. Clone the repo and enter blatt02/  (set URL/branch/subdir here)
REPO_URL = "https://github.com/JDeffner/Heterogeneous-Computing.git"
BRANCH = "main"
SUBDIR = "jdeffner-HC02/blatt02"  # path of blatt02/ inside the repo

import os
repo_dir = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if not os.path.isdir(repo_dir):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL}
%cd {repo_dir}/{SUBDIR}

In [ ]:
# 3. Build
!cmake -B build -DCMAKE_BUILD_TYPE=Release && cmake --build build -j

In [ ]:
# 3b. Fallback: raw nvcc/g++ build, only needed if the CMake cell failed
if False:  # flip to True to use the fallback
    !mkdir -p build
    !nvcc -O3 -std=c++17 -arch=native -Xcompiler -Wall,-Wextra src/task1_divergence.cu -o build/task1_divergence
    !nvcc -O3 -std=c++17 -arch=native -Xcompiler -Wall,-Wextra src/task2_bandwidth.cu -o build/task2_bandwidth
    !g++ -O3 -std=c++17 -march=native -fopenmp -Wall -Wextra src/cpu_baseline.cpp -o build/cpu_baseline

In [ ]:
# 4. Run all sweeps (writes results/*.csv)
!bash scripts/run_all.sh

In [ ]:
# 5. Plots (writes results/plots/*.png) and summary numbers
!python scripts/plot.py

In [ ]:
# 6. Zip results/ and download
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")

Optional: verify the divergence experiment (spec 3.3). Register/instruction stats:
`!nvcc -O3 -std=c++17 -arch=native --ptxas-options=-v -c src/task1_divergence.cu -o /dev/null`

Measured occupancy via Nsight Compute usually fails on Colab (counter permissions), but you can try:
`!ncu --metrics sm__warps_active.avg.pct_of_peak_sustained_active build/task2_bandwidth --exp occupancy`